Muril

In [1]:
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL_PATH = r'D:\Major Project\SpamX\machine_learning\models\MuRIL\final_muril'
device = "cuda" if torch.cuda.is_available() else "cpu"
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH).to(device)
model.eval()

stress_samples = [
    {"text": "ഈ വീഡിയോ കണ്ടിട്ട് എനിക്ക് ഒരുപാട് ഇഷ്ടമായി, സൂപ്പർ!", "label": "HAM"},
    {"text": "സൗജന്യമായി പണം സമ്പാദിക്കാൻ ഈ ലിങ്കിൽ ക്ലിക്ക് ചെയ്യുക: bit.ly/scam", "label": "SPAM"},
    
    {"text": "Aliyaa ithu kidu sadhanam aanu, thakarppan video!", "label": "HAM"},
    {"text": "Ente profile onnu keri noku, giveaway undu guys", "label": "SPAM"},
    
    {"text": "Incredible explanation, helped me a lot for my exams.", "label": "HAM"},
    {"text": "GET 1000 SUBS FAST! NO SCAM! VISIT MY CHANNEL NOW!", "label": "SPAM"},
    
    {"text": "Super video aliyaa, keep it up! ❤️", "label": "HAM"},
    {"text": "Click here for free coins/സൗജന്യ കോയിനുകൾക്കായി ഇവിടെ ക്ലിക്ക് ചെയ്യുക", "label": "SPAM"}
]

print(f"Test on MuRIL...")
print("-" * 80)

for sample in stress_samples:
    inputs = tokenizer(sample['text'], return_tensors="pt", truncation=True, max_length=128).to(device)
    with torch.no_grad():
        outputs = model(**inputs)
        probs = F.softmax(outputs.logits, dim=-1).cpu().numpy()[0]
        pred = torch.argmax(outputs.logits, dim=-1).item()
    
    status = "PASS" if (pred == 1 and sample['label'] == "SPAM") or (pred == 0 and sample['label'] == "HAM") else "FAIL"
    print(f"Text: {sample['text'][:50]}...")
    print(f"Pred: {'SPAM' if pred==1 else 'HAM'} | Conf: {probs[pred]*100:.2f}% | {status}\n")

Test on MuRIL...
--------------------------------------------------------------------------------
Text: ഈ വീഡിയോ കണ്ടിട്ട് എനിക്ക് ഒരുപാട് ഇഷ്ടമായി, സൂപ്പ...
Pred: HAM | Conf: 89.34% | PASS

Text: സൗജന്യമായി പണം സമ്പാദിക്കാൻ ഈ ലിങ്കിൽ ക്ലിക്ക് ചെയ...
Pred: SPAM | Conf: 96.21% | PASS

Text: Aliyaa ithu kidu sadhanam aanu, thakarppan video!...
Pred: HAM | Conf: 85.65% | PASS

Text: Ente profile onnu keri noku, giveaway undu guys...
Pred: SPAM | Conf: 72.10% | PASS

Text: Incredible explanation, helped me a lot for my exa...
Pred: HAM | Conf: 90.44% | PASS

Text: GET 1000 SUBS FAST! NO SCAM! VISIT MY CHANNEL NOW!...
Pred: SPAM | Conf: 90.11% | PASS

Text: Super video aliyaa, keep it up! ❤️...
Pred: HAM | Conf: 88.65% | PASS

Text: Click here for free coins/സൗജന്യ കോയിനുകൾക്കായി ഇവ...
Pred: SPAM | Conf: 95.42% | PASS



In [2]:
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix

TEST_PATH = r'D:\Major Project\SpamX\machine_learning\dataset\test.csv'
test_df = pd.read_csv(TEST_PATH).fillna("")

print(f"Evaluating on {len(test_df)} Test Samples...")

all_preds = []
all_labels = test_df['Label'].tolist()

for text in test_df['Comment']:
    inputs = tokenizer(str(text), return_tensors="pt", truncation=True, max_length=128).to(device)
    with torch.no_grad():
        logits = model(**inputs).logits
        all_preds.append(torch.argmax(logits, dim=-1).item())

print("\nMuRIL PERFORMANCE REPORT")
print("-" * 60)
print(classification_report(all_labels, all_preds, target_names=['HAM', 'SPAM']))

Evaluating on 6438 Test Samples...

MuRIL PERFORMANCE REPORT
------------------------------------------------------------
              precision    recall  f1-score   support

         HAM       0.96      0.99      0.97      3219
        SPAM       0.99      0.96      0.97      3219

    accuracy                           0.97      6438
   macro avg       0.97      0.97      0.97      6438
weighted avg       0.97      0.97      0.97      6438



XLM-ROBERTA

In [3]:
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL_PATH = r'D:\Major Project\SpamX\machine_learning\models\XLM_Roberta\final_xlm_roberta'
device = "cuda" if torch.cuda.is_available() else "cpu"
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH).to(device)
model.eval()

stress_samples = [
    {"text": "ഈ വീഡിയോ കണ്ടിട്ട് എനിക്ക് ഒരുപാട് ഇഷ്ടമായി, സൂപ്പർ!", "label": "HAM"},
    {"text": "സൗജന്യമായി പണം സമ്പാദിക്കാൻ ഈ ലിങ്കിൽ ക്ലിക്ക് ചെയ്യുക: bit.ly/scam", "label": "SPAM"},
    
    {"text": "Aliyaa ithu kidu sadhanam aanu, thakarppan video!", "label": "HAM"},
    {"text": "Ente profile onnu keri noku, giveaway undu guys", "label": "SPAM"},
    
    {"text": "Incredible explanation, helped me a lot for my exams.", "label": "HAM"},
    {"text": "GET 1000 SUBS FAST! NO SCAM! VISIT MY CHANNEL NOW!", "label": "SPAM"},
    
    {"text": "Super video aliyaa, keep it up! ❤️", "label": "HAM"},
    {"text": "Click here for free coins/സൗജന്യ കോയിനുകൾക്കായി ഇവിടെ ക്ലിക്ക് ചെയ്യുക", "label": "SPAM"}
]

print(f"Test on XLM RoBERTa...")
print("-" * 80)

for sample in stress_samples:
    inputs = tokenizer(sample['text'], return_tensors="pt", truncation=True, max_length=128).to(device)
    with torch.no_grad():
        outputs = model(**inputs)
        probs = F.softmax(outputs.logits, dim=-1).cpu().numpy()[0]
        pred = torch.argmax(outputs.logits, dim=-1).item()
    
    status = "PASS" if (pred == 1 and sample['label'] == "SPAM") or (pred == 0 and sample['label'] == "HAM") else "FAIL"
    print(f"Text: {sample['text'][:50]}...")
    print(f"Pred: {'SPAM' if pred==1 else 'HAM'} | Conf: {probs[pred]*100:.2f}% | {status}\n")

Test on XLM RoBERTa...
--------------------------------------------------------------------------------
Text: ഈ വീഡിയോ കണ്ടിട്ട് എനിക്ക് ഒരുപാട് ഇഷ്ടമായി, സൂപ്പ...
Pred: HAM | Conf: 86.71% | PASS

Text: സൗജന്യമായി പണം സമ്പാദിക്കാൻ ഈ ലിങ്കിൽ ക്ലിക്ക് ചെയ...
Pred: SPAM | Conf: 91.53% | PASS

Text: Aliyaa ithu kidu sadhanam aanu, thakarppan video!...
Pred: HAM | Conf: 83.04% | PASS

Text: Ente profile onnu keri noku, giveaway undu guys...
Pred: SPAM | Conf: 69.46% | PASS

Text: Incredible explanation, helped me a lot for my exa...
Pred: HAM | Conf: 89.85% | PASS

Text: GET 1000 SUBS FAST! NO SCAM! VISIT MY CHANNEL NOW!...
Pred: SPAM | Conf: 92.49% | PASS

Text: Super video aliyaa, keep it up! ❤️...
Pred: HAM | Conf: 86.80% | PASS

Text: Click here for free coins/സൗജന്യ കോയിനുകൾക്കായി ഇവ...
Pred: SPAM | Conf: 80.40% | PASS



In [4]:
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix

TEST_PATH = r'D:\Major Project\SpamX\machine_learning\dataset\test.csv'
test_df = pd.read_csv(TEST_PATH).fillna("")

print(f"Evaluating on {len(test_df)} Test Samples...")

all_preds = []
all_labels = test_df['Label'].tolist()

for text in test_df['Comment']:
    inputs = tokenizer(str(text), return_tensors="pt", truncation=True, max_length=128).to(device)
    with torch.no_grad():
        logits = model(**inputs).logits
        all_preds.append(torch.argmax(logits, dim=-1).item())

print("\nXLM RoBERTa PERFORMANCE REPORT")
print("-" * 60)
print(classification_report(all_labels, all_preds, target_names=['HAM', 'SPAM']))

Evaluating on 6438 Test Samples...

XLM RoBERTa PERFORMANCE REPORT
------------------------------------------------------------
              precision    recall  f1-score   support

         HAM       0.92      1.00      0.96      3219
        SPAM       1.00      0.91      0.95      3219

    accuracy                           0.95      6438
   macro avg       0.96      0.95      0.95      6438
weighted avg       0.96      0.95      0.95      6438



Indic BERT

In [5]:
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL_PATH = r'D:\Major Project\SpamX\machine_learning\models\IndicBERT\final_indicbert'
device = "cuda" if torch.cuda.is_available() else "cpu"
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH).to(device)
model.eval()

stress_samples = [
    {"text": "ഈ വീഡിയോ കണ്ടിട്ട് എനിക്ക് ഒരുപാട് ഇഷ്ടമായി, സൂപ്പർ!", "label": "HAM"},
    {"text": "സൗജന്യമായി പണം സമ്പാദിക്കാൻ ഈ ലിങ്കിൽ ക്ലിക്ക് ചെയ്യുക: bit.ly/scam", "label": "SPAM"},
    
    {"text": "Aliyaa ithu kidu sadhanam aanu, thakarppan video!", "label": "HAM"},
    {"text": "Ente profile onnu keri noku, giveaway undu guys", "label": "SPAM"},
    
    {"text": "Incredible explanation, helped me a lot for my exams.", "label": "HAM"},
    {"text": "GET 1000 SUBS FAST! NO SCAM! VISIT MY CHANNEL NOW!", "label": "SPAM"},
    
    {"text": "Super video aliyaa, keep it up! ❤️", "label": "HAM"},
    {"text": "Click here for free coins/സൗജന്യ കോയിനുകൾക്കായി ഇവിടെ ക്ലിക്ക് ചെയ്യുക", "label": "SPAM"}
]

print(f"Test on Indic BERT...")
print("-" * 80)

for sample in stress_samples:
    inputs = tokenizer(sample['text'], return_tensors="pt", truncation=True, max_length=128).to(device)
    with torch.no_grad():
        outputs = model(**inputs)
        probs = F.softmax(outputs.logits, dim=-1).cpu().numpy()[0]
        pred = torch.argmax(outputs.logits, dim=-1).item()
    
    status = "PASS" if (pred == 1 and sample['label'] == "SPAM") or (pred == 0 and sample['label'] == "HAM") else "FAIL"
    print(f"Text: {sample['text'][:50]}...")
    print(f"Pred: {'SPAM' if pred==1 else 'HAM'} | Conf: {probs[pred]*100:.2f}% | {status}\n")

Test on Indic BERT...
--------------------------------------------------------------------------------
Text: ഈ വീഡിയോ കണ്ടിട്ട് എനിക്ക് ഒരുപാട് ഇഷ്ടമായി, സൂപ്പ...
Pred: HAM | Conf: 88.66% | PASS

Text: സൗജന്യമായി പണം സമ്പാദിക്കാൻ ഈ ലിങ്കിൽ ക്ലിക്ക് ചെയ...
Pred: SPAM | Conf: 94.95% | PASS

Text: Aliyaa ithu kidu sadhanam aanu, thakarppan video!...
Pred: HAM | Conf: 90.48% | PASS

Text: Ente profile onnu keri noku, giveaway undu guys...
Pred: HAM | Conf: 52.50% | FAIL

Text: Incredible explanation, helped me a lot for my exa...
Pred: HAM | Conf: 94.24% | PASS

Text: GET 1000 SUBS FAST! NO SCAM! VISIT MY CHANNEL NOW!...
Pred: SPAM | Conf: 80.53% | PASS

Text: Super video aliyaa, keep it up! ❤️...
Pred: HAM | Conf: 89.57% | PASS

Text: Click here for free coins/സൗജന്യ കോയിനുകൾക്കായി ഇവ...
Pred: SPAM | Conf: 89.14% | PASS



In [6]:
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix

TEST_PATH = r'D:\Major Project\SpamX\machine_learning\dataset\test.csv'
test_df = pd.read_csv(TEST_PATH).fillna("")

print(f"Evaluating on {len(test_df)} Test Samples...")

all_preds = []
all_labels = test_df['Label'].tolist()

for text in test_df['Comment']:
    inputs = tokenizer(str(text), return_tensors="pt", truncation=True, max_length=128).to(device)
    with torch.no_grad():
        logits = model(**inputs).logits
        all_preds.append(torch.argmax(logits, dim=-1).item())

print("\nIndicBERT PERFORMANCE REPORT")
print("-" * 60)
print(classification_report(all_labels, all_preds, target_names=['HAM', 'SPAM']))

Evaluating on 6438 Test Samples...

IndicBERT PERFORMANCE REPORT
------------------------------------------------------------
              precision    recall  f1-score   support

         HAM       0.95      0.99      0.97      3219
        SPAM       0.99      0.95      0.97      3219

    accuracy                           0.97      6438
   macro avg       0.97      0.97      0.97      6438
weighted avg       0.97      0.97      0.97      6438



mBERT

In [1]:
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL_PATH = r'D:\Major Project\SpamX\machine_learning\models\mBERT\final_mbert'
device = "cuda" if torch.cuda.is_available() else "cpu"
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH).to(device)
model.eval()

stress_samples = [
    {"text": "ഈ വീഡിയോ കണ്ടിട്ട് എനിക്ക് ഒരുപാട് ഇഷ്ടമായി, സൂപ്പർ!", "label": "HAM"},
    {"text": "സൗജന്യമായി പണം സമ്പാദിക്കാൻ ഈ ലിങ്കിൽ ക്ലിക്ക് ചെയ്യുക: bit.ly/scam", "label": "SPAM"},
    
    {"text": "Aliyaa ithu kidu sadhanam aanu, thakarppan video!", "label": "HAM"},
    {"text": "Ente profile onnu keri noku, giveaway undu guys", "label": "SPAM"},
    
    {"text": "Incredible explanation, helped me a lot for my exams.", "label": "HAM"},
    {"text": "GET 1000 SUBS FAST! NO SCAM! VISIT MY CHANNEL NOW!", "label": "SPAM"},
    
    {"text": "Super video aliyaa, keep it up! ❤️", "label": "HAM"},
    {"text": "Click here for free coins/സൗജന്യ കോയിനുകൾക്കായി ഇവിടെ ക്ലിക്ക് ചെയ്യുക", "label": "SPAM"}
]

print(f"Test on mBERT...")
print("-" * 80)

for sample in stress_samples:
    inputs = tokenizer(sample['text'], return_tensors="pt", truncation=True, max_length=128).to(device)
    with torch.no_grad():
        outputs = model(**inputs)
        probs = F.softmax(outputs.logits, dim=-1).cpu().numpy()[0]
        pred = torch.argmax(outputs.logits, dim=-1).item()
    
    status = "PASS" if (pred == 1 and sample['label'] == "SPAM") or (pred == 0 and sample['label'] == "HAM") else "FAIL"
    print(f"Text: {sample['text'][:50]}...")
    print(f"Pred: {'SPAM' if pred==1 else 'HAM'} | Conf: {probs[pred]*100:.2f}% | {status}\n")

Test on mBERT...
--------------------------------------------------------------------------------
Text: ഈ വീഡിയോ കണ്ടിട്ട് എനിക്ക് ഒരുപാട് ഇഷ്ടമായി, സൂപ്പ...
Pred: HAM | Conf: 84.82% | PASS

Text: സൗജന്യമായി പണം സമ്പാദിക്കാൻ ഈ ലിങ്കിൽ ക്ലിക്ക് ചെയ...
Pred: SPAM | Conf: 96.30% | PASS

Text: Aliyaa ithu kidu sadhanam aanu, thakarppan video!...
Pred: HAM | Conf: 86.90% | PASS

Text: Ente profile onnu keri noku, giveaway undu guys...
Pred: HAM | Conf: 81.26% | FAIL

Text: Incredible explanation, helped me a lot for my exa...
Pred: HAM | Conf: 85.89% | PASS

Text: GET 1000 SUBS FAST! NO SCAM! VISIT MY CHANNEL NOW!...
Pred: SPAM | Conf: 86.76% | PASS

Text: Super video aliyaa, keep it up! ❤️...
Pred: HAM | Conf: 86.28% | PASS

Text: Click here for free coins/സൗജന്യ കോയിനുകൾക്കായി ഇവ...
Pred: SPAM | Conf: 73.74% | PASS



In [2]:
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix

TEST_PATH = r'D:\Major Project\SpamX\machine_learning\dataset\test.csv'
test_df = pd.read_csv(TEST_PATH).fillna("")

print(f"Evaluating on {len(test_df)} Test Samples...")

all_preds = []
all_labels = test_df['Label'].tolist()

for text in test_df['Comment']:
    inputs = tokenizer(str(text), return_tensors="pt", truncation=True, max_length=128).to(device)
    with torch.no_grad():
        logits = model(**inputs).logits
        all_preds.append(torch.argmax(logits, dim=-1).item())

print("\nmBERT PERFORMANCE REPORT")
print("-" * 60)
print(classification_report(all_labels, all_preds, target_names=['HAM', 'SPAM']))

Evaluating on 6438 Test Samples...

mBERT PERFORMANCE REPORT
------------------------------------------------------------
              precision    recall  f1-score   support

         HAM       0.96      0.99      0.97      3219
        SPAM       0.99      0.96      0.97      3219

    accuracy                           0.97      6438
   macro avg       0.97      0.97      0.97      6438
weighted avg       0.97      0.97      0.97      6438



Calibrated MuRIL

In [1]:
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL_PATH = r'D:\Major Project\SpamX\machine_learning\models\MuRIL\final_muril_calibrated1'
device = "cuda" if torch.cuda.is_available() else "cpu"
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH).to(device)
model.eval()

stress_samples = [
    {"text": "ഈ വീഡിയോ കണ്ടിട്ട് എനിക്ക് ഒരുപാട് ഇഷ്ടമായി, സൂപ്പർ!", "label": "HAM"},
    {"text": "സൗജന്യമായി പണം സമ്പാദിക്കാൻ ഈ ലിങ്കിൽ ക്ലിക്ക് ചെയ്യുക: bit.ly/scam", "label": "SPAM"},
    
    {"text": "Aliyaa ithu kidu sadhanam aanu, thakarppan video!", "label": "HAM"},
    {"text": "Ente profile onnu keri noku, giveaway undu guys", "label": "SPAM"},
    
    {"text": "Incredible explanation, helped me a lot for my exams.", "label": "HAM"},
    {"text": "GET 1000 SUBS FAST! NO SCAM! VISIT MY CHANNEL NOW!", "label": "SPAM"},
    
    {"text": "Super video aliyaa, keep it up! ❤️", "label": "HAM"},
    {"text": "Click here for free coins/സൗജന്യ കോയിനുകൾക്കായി ഇവിടെ ക്ലിക്ക് ചെയ്യുക", "label": "SPAM"}
]

print(f"Test on MuRIL...")
print("-" * 80)

for sample in stress_samples:
    inputs = tokenizer(sample['text'], return_tensors="pt", truncation=True, max_length=128).to(device)
    with torch.no_grad():
        outputs = model(**inputs)
        probs = F.softmax(outputs.logits, dim=-1).cpu().numpy()[0]
        pred = torch.argmax(outputs.logits, dim=-1).item()
    
    status = "PASS" if (pred == 1 and sample['label'] == "SPAM") or (pred == 0 and sample['label'] == "HAM") else "FAIL"
    print(f"Text: {sample['text'][:50]}...")
    print(f"Pred: {'SPAM' if pred==1 else 'HAM'} | Conf: {probs[pred]*100:.2f}% | {status}\n")

Test on MuRIL...
--------------------------------------------------------------------------------
Text: ഈ വീഡിയോ കണ്ടിട്ട് എനിക്ക് ഒരുപാട് ഇഷ്ടമായി, സൂപ്പ...
Pred: HAM | Conf: 83.70% | PASS

Text: സൗജന്യമായി പണം സമ്പാദിക്കാൻ ഈ ലിങ്കിൽ ക്ലിക്ക് ചെയ...
Pred: SPAM | Conf: 96.10% | PASS

Text: Aliyaa ithu kidu sadhanam aanu, thakarppan video!...
Pred: HAM | Conf: 71.26% | PASS

Text: Ente profile onnu keri noku, giveaway undu guys...
Pred: SPAM | Conf: 88.87% | PASS

Text: Incredible explanation, helped me a lot for my exa...
Pred: HAM | Conf: 87.66% | PASS

Text: GET 1000 SUBS FAST! NO SCAM! VISIT MY CHANNEL NOW!...
Pred: SPAM | Conf: 93.64% | PASS

Text: Super video aliyaa, keep it up! ❤️...
Pred: HAM | Conf: 70.35% | PASS

Text: Click here for free coins/സൗജന്യ കോയിനുകൾക്കായി ഇവ...
Pred: SPAM | Conf: 95.25% | PASS



In [2]:
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix

TEST_PATH = r'D:\Major Project\SpamX\machine_learning\dataset\test.csv'
test_df = pd.read_csv(TEST_PATH).fillna("")

print(f"Evaluating on {len(test_df)} Test Samples...")

all_preds = []
all_labels = test_df['Label'].tolist()

for text in test_df['Comment']:
    inputs = tokenizer(str(text), return_tensors="pt", truncation=True, max_length=128).to(device)
    with torch.no_grad():
        logits = model(**inputs).logits
        all_preds.append(torch.argmax(logits, dim=-1).item())

print("\nMuRIL PERFORMANCE REPORT")
print("-" * 60)
print(classification_report(all_labels, all_preds, target_names=['HAM', 'SPAM']))

Evaluating on 6438 Test Samples...

MuRIL PERFORMANCE REPORT
------------------------------------------------------------
              precision    recall  f1-score   support

         HAM       0.97      0.97      0.97      3219
        SPAM       0.97      0.97      0.97      3219

    accuracy                           0.97      6438
   macro avg       0.97      0.97      0.97      6438
weighted avg       0.97      0.97      0.97      6438



Calibrated XLM RoBERTa

In [3]:
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL_PATH = r'D:\Major Project\SpamX\machine_learning\models\XLM_Roberta\final_xlm_roberta_calibrated1'
device = "cuda" if torch.cuda.is_available() else "cpu"
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH).to(device)
model.eval()

stress_samples = [
    {"text": "ഈ വീഡിയോ കണ്ടിട്ട് എനിക്ക് ഒരുപാട് ഇഷ്ടമായി, സൂപ്പർ!", "label": "HAM"},
    {"text": "സൗജന്യമായി പണം സമ്പാദിക്കാൻ ഈ ലിങ്കിൽ ക്ലിക്ക് ചെയ്യുക: bit.ly/scam", "label": "SPAM"},
    
    {"text": "Aliyaa ithu kidu sadhanam aanu, thakarppan video!", "label": "HAM"},
    {"text": "Ente profile onnu keri noku, giveaway undu guys", "label": "SPAM"},
    
    {"text": "Incredible explanation, helped me a lot for my exams.", "label": "HAM"},
    {"text": "GET 1000 SUBS FAST! NO SCAM! VISIT MY CHANNEL NOW!", "label": "SPAM"},
    
    {"text": "Super video aliyaa, keep it up! ❤️", "label": "HAM"},
    {"text": "Click here for free coins/സൗജന്യ കോയിനുകൾക്കായി ഇവിടെ ക്ലിക്ക് ചെയ്യുക", "label": "SPAM"}
]

print(f"Test on XLM RoBERTa...")
print("-" * 80)

for sample in stress_samples:
    inputs = tokenizer(sample['text'], return_tensors="pt", truncation=True, max_length=128).to(device)
    with torch.no_grad():
        outputs = model(**inputs)
        probs = F.softmax(outputs.logits, dim=-1).cpu().numpy()[0]
        pred = torch.argmax(outputs.logits, dim=-1).item()
    
    status = "PASS" if (pred == 1 and sample['label'] == "SPAM") or (pred == 0 and sample['label'] == "HAM") else "FAIL"
    print(f"Text: {sample['text'][:50]}...")
    print(f"Pred: {'SPAM' if pred==1 else 'HAM'} | Conf: {probs[pred]*100:.2f}% | {status}\n")

Test on XLM RoBERTa...
--------------------------------------------------------------------------------
Text: ഈ വീഡിയോ കണ്ടിട്ട് എനിക്ക് ഒരുപാട് ഇഷ്ടമായി, സൂപ്പ...
Pred: HAM | Conf: 80.98% | PASS

Text: സൗജന്യമായി പണം സമ്പാദിക്കാൻ ഈ ലിങ്കിൽ ക്ലിക്ക് ചെയ...
Pred: SPAM | Conf: 92.98% | PASS

Text: Aliyaa ithu kidu sadhanam aanu, thakarppan video!...
Pred: HAM | Conf: 66.05% | PASS

Text: Ente profile onnu keri noku, giveaway undu guys...
Pred: SPAM | Conf: 86.85% | PASS

Text: Incredible explanation, helped me a lot for my exa...
Pred: HAM | Conf: 86.05% | PASS

Text: GET 1000 SUBS FAST! NO SCAM! VISIT MY CHANNEL NOW!...
Pred: SPAM | Conf: 92.77% | PASS

Text: Super video aliyaa, keep it up! ❤️...
Pred: HAM | Conf: 81.30% | PASS

Text: Click here for free coins/സൗജന്യ കോയിനുകൾക്കായി ഇവ...
Pred: SPAM | Conf: 84.92% | PASS



In [4]:
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix

TEST_PATH = r'D:\Major Project\SpamX\machine_learning\dataset\test.csv'
test_df = pd.read_csv(TEST_PATH).fillna("")

print(f"Evaluating on {len(test_df)} Test Samples...")

all_preds = []
all_labels = test_df['Label'].tolist()

for text in test_df['Comment']:
    inputs = tokenizer(str(text), return_tensors="pt", truncation=True, max_length=128).to(device)
    with torch.no_grad():
        logits = model(**inputs).logits
        all_preds.append(torch.argmax(logits, dim=-1).item())

print("\nXLM RoBERTa PERFORMANCE REPORT")
print("-" * 60)
print(classification_report(all_labels, all_preds, target_names=['HAM', 'SPAM']))

Evaluating on 6438 Test Samples...

XLM RoBERTa PERFORMANCE REPORT
------------------------------------------------------------
              precision    recall  f1-score   support

         HAM       0.94      0.97      0.96      3219
        SPAM       0.97      0.94      0.96      3219

    accuracy                           0.96      6438
   macro avg       0.96      0.96      0.96      6438
weighted avg       0.96      0.96      0.96      6438



Calibrated XLM RoBERTa V2

In [2]:
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix

TEST_PATH = r'D:\Major Project\SpamX\machine_learning\dataset\test.csv'
test_df = pd.read_csv(TEST_PATH).fillna("")

print(f"Evaluating on {len(test_df)} Test Samples...")

all_preds = []
all_labels = test_df['Label'].tolist()

for text in test_df['Comment']:
    inputs = tokenizer(str(text), return_tensors="pt", truncation=True, max_length=128).to(device)
    with torch.no_grad():
        logits = model(**inputs).logits
        all_preds.append(torch.argmax(logits, dim=-1).item())

print("\nXLM RoBERTa PERFORMANCE REPORT")
print("-" * 60)
print(classification_report(all_labels, all_preds, target_names=['HAM', 'SPAM']))

Evaluating on 6438 Test Samples...

XLM RoBERTa PERFORMANCE REPORT
------------------------------------------------------------
              precision    recall  f1-score   support

         HAM       0.96      0.98      0.97      3219
        SPAM       0.98      0.96      0.97      3219

    accuracy                           0.97      6438
   macro avg       0.97      0.97      0.97      6438
weighted avg       0.97      0.97      0.97      6438



In [4]:
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL_PATH = r'D:\Major Project\SpamX\machine_learning\models\MuRIL\final_muril_calibrated1_v2'
device = "cuda" if torch.cuda.is_available() else "cpu"
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH).to(device)
model.eval()

stress_samples = [
    {"text": "ഈ വീഡിയോ കണ്ടിട്ട് എനിക്ക് ഒരുപാട് ഇഷ്ടമായി, സൂപ്പർ!", "label": "HAM"},
    {"text": "സൗജന്യമായി പണം സമ്പാദിക്കാൻ ഈ ലിങ്കിൽ ക്ലിക്ക് ചെയ്യുക: bit.ly/scam", "label": "SPAM"},
    
    {"text": "Aliyaa ithu kidu sadhanam aanu, thakarppan video!", "label": "HAM"},
    {"text": "Ente profile onnu keri noku, giveaway undu guys", "label": "SPAM"},
    
    {"text": "Incredible explanation, helped me a lot for my exams.", "label": "HAM"},
    {"text": "GET 1000 SUBS FAST! NO SCAM! VISIT MY CHANNEL NOW!", "label": "SPAM"},
    
    {"text": "Super video aliyaa, keep it up! ❤️", "label": "HAM"},
    {"text": "Click here for free coins/സൗജന്യ കോയിനുകൾക്കായി ഇവിടെ ക്ലിക്ക് ചെയ്യുക", "label": "SPAM"}
]

print(f"Test on MuRIL...")
print("-" * 80)

for sample in stress_samples:
    inputs = tokenizer(sample['text'], return_tensors="pt", truncation=True, max_length=128).to(device)
    with torch.no_grad():
        outputs = model(**inputs)
        probs = F.softmax(outputs.logits, dim=-1).cpu().numpy()[0]
        pred = torch.argmax(outputs.logits, dim=-1).item()
    
    status = "PASS" if (pred == 1 and sample['label'] == "SPAM") or (pred == 0 and sample['label'] == "HAM") else "FAIL"
    print(f"Text: {sample['text'][:50]}...")
    print(f"Pred: {'SPAM' if pred==1 else 'HAM'} | Conf: {probs[pred]*100:.2f}% | {status}\n")

Test on MuRIL...
--------------------------------------------------------------------------------
Text: ഈ വീഡിയോ കണ്ടിട്ട് എനിക്ക് ഒരുപാട് ഇഷ്ടമായി, സൂപ്പ...
Pred: HAM | Conf: 89.28% | PASS

Text: സൗജന്യമായി പണം സമ്പാദിക്കാൻ ഈ ലിങ്കിൽ ക്ലിക്ക് ചെയ...
Pred: SPAM | Conf: 96.11% | PASS

Text: Aliyaa ithu kidu sadhanam aanu, thakarppan video!...
Pred: HAM | Conf: 86.89% | PASS

Text: Ente profile onnu keri noku, giveaway undu guys...
Pred: SPAM | Conf: 91.10% | PASS

Text: Incredible explanation, helped me a lot for my exa...
Pred: HAM | Conf: 89.69% | PASS

Text: GET 1000 SUBS FAST! NO SCAM! VISIT MY CHANNEL NOW!...
Pred: SPAM | Conf: 55.14% | PASS

Text: Super video aliyaa, keep it up! ❤️...
Pred: HAM | Conf: 89.55% | PASS

Text: Click here for free coins/സൗജന്യ കോയിനുകൾക്കായി ഇവ...
Pred: SPAM | Conf: 96.11% | PASS



In [5]:
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix

TEST_PATH = r'D:\Major Project\SpamX\machine_learning\dataset\test.csv'
test_df = pd.read_csv(TEST_PATH).fillna("")

print(f"Evaluating on {len(test_df)} Test Samples...")

all_preds = []
all_labels = test_df['Label'].tolist()

for text in test_df['Comment']:
    inputs = tokenizer(str(text), return_tensors="pt", truncation=True, max_length=128).to(device)
    with torch.no_grad():
        logits = model(**inputs).logits
        all_preds.append(torch.argmax(logits, dim=-1).item())

print("\nMuRIL PERFORMANCE REPORT")
print("-" * 60)
print(classification_report(all_labels, all_preds, target_names=['HAM', 'SPAM']))

Evaluating on 6438 Test Samples...

MuRIL PERFORMANCE REPORT
------------------------------------------------------------
              precision    recall  f1-score   support

         HAM       0.97      0.98      0.97      3219
        SPAM       0.98      0.97      0.97      3219

    accuracy                           0.97      6438
   macro avg       0.97      0.97      0.97      6438
weighted avg       0.97      0.97      0.97      6438

